# population

In [ ]:
#| default_exp game/flag

In [ ]:
#| export
from fastcore.basics import patch

# fun with colors
import colorsys
import seaborn as sns
import matplotlib.pyplot as plt
from importlib import resources
import pandas as pd
import random
import re

In [ ]:
#| export
from HexMagic.styles import   SVGBuilder, SVGDef,  Generatable, NamedColor, StyleCSS
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord , Hex, HexGrid, PrimitiveDemo
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion ,  unique_windy_edge
from HexMagic.terrainpatterns import TerrainPatterns, PathPattern
from HexMagic.styles import apply_looping_animation, LoopingLayerAnimation
from HexMagic.terrain import Terrain


In [ ]:
#| export
class CountryFlag:

    def __init__(self, c, name, year,patternIndex = 0):
        self.primary = plt.matplotlib.colors.rgb2hex(c)
        h, s, v = colorsys.rgb_to_hsv(*c)
        self.comp = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb((h + 0.5) % 1, s, v))
        self.tri1 = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb((h + 1/3) % 1, s, v))
        self.tri2 = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb((h - 1/3) % 1, s, v))
        self.name = name
        self.year = year
        self.countryPrefix = CountryFlag.countryName(name)
        self.capital = CountryFlag.cityName(name)
        self.patternIndex = patternIndex

    @classmethod
    def seaborn(cls,name:str,levels=7,gender=None,year=1900):
        palette = sns.color_palette(name, levels)

        if gender is not None:
            names = random.sample(CountryFlag.commonNames(year=year,gender=gender),levels)
        else:
            allN = CountryFlag.commonNames(year=year,gender="M") + CountryFlag.commonNames(year=year,gender="F")
            names = random.sample(allN,levels)
            
       
        ret = []
        for i, color in enumerate(palette):
            ret.append(cls(color,name=names[i],year=year))
        return ret

    
    @classmethod
    def commonNames(cls,year=1900,gender="M"):
        
        with resources.files('HexMagic').joinpath('data/misc/popularNames.csv').open() as f:
            df = pd.read_csv(f)
        
        filtered = df[(df['Year'] == year) & (df['Gender'] == gender)]
        if filtered.empty:
            raise ValueError(f"No names found for year={year}, gender={gender}")
        
        # Weight by count for more realistic distribution
        return filtered['Name'].tolist()


    @classmethod
    def countryName(cls, name , pattern=None, descriptor=None):
        # Dictionary of place descriptors organized by first letter
        PLACE_DESCRIPTORS = {
            'A': ['Abbey', 'Acres', 'Alcove', 'Apex', 'Archipelago', 'Arena', 'Atoll', 'Avenue'],
            'B': ['Basin', 'Bay', 'Bluff', 'Borough', 'Boundary', 'Bower', 'Burg', 'Borderlands'],
            'C': ['Canyon', 'Cape', 'Castle', 'Citadel', 'Clearing', 'Cove', 'Crossing', 'County'],
            'D': ['Dale', 'Dell', 'Delta', 'Den', 'District', 'Domain', 'Dunes', 'Dominion'],
            'E': ['Edge', 'Enclave', 'End', 'Estate', 'Expanse', 'Empire', 'Escarpment', 'Eyrie'],
            'F': ['Falls', 'Fen', 'Fjord', 'Forest', 'Fort', 'Frontier', 'Fields', 'Fief'],
            'G': ['Gap', 'Garden', 'Gate', 'Glade', 'Glen', 'Gorge', 'Grotto', 'Grove'],
            'H': ['Habitat', 'Harbor', 'Haven', 'Heath', 'Heights', 'Hideaway', 'Hill', 'Hollow'],
            'I': ['Isle', 'Inlet', 'Island', 'Isthmus', 'Ironworks', 'Imperium', 'Inn', 'Impasse'],
            'J': ['Junction', 'Jungle', 'Jetty', 'Juncture', 'Jurisdiction', 'Jut', 'Joint', 'Jewel'],
            'K': ['Keep', 'Kingdom', 'Knoll', 'Key', 'Knot', 'Kiosk', 'Krantz', 'Karst'],
            'L': ['Lagoon', 'Lake', 'Landing', 'Land', 'Lair', 'Ledge', 'Lodge', 'Lowlands'],
            'M': ['Manor', 'Marsh', 'Meadow', 'Mesa', 'Moor', 'Mount', 'Mountains', 'Mound'],
            'N': ['Narrows', 'Nest', 'Niche', 'Nook', 'North', 'Notch', 'Nation', 'Neighborhood'],
            'O': ['Oasis', 'Observatory', 'Outpost', 'Overlook', 'Orchard', 'Outcrop', 'Outlet', 'Outlands'],
            'P': ['Palace', 'Park', 'Pass', 'Path', 'Peak', 'Peninsula', 'Pinnacle', 'Plaza', 'Point', 'Province'],
            'Q': ['Quarry', 'Quarter', 'Quay', 'Quarters', 'Quad', 'Quadrant', 'Quest', 'Quietude'],
            'R': ['Range', 'Ravine', 'Reach', 'Realm', 'Reef', 'Region', 'Reserve', 'Retreat', 'Ridge', 'Rise'],
            'S': ['Sanctuary', 'Settlement', 'Shire', 'Shore', 'Slopes', 'Sound', 'Span', 'Spring', 'Summit', 'Stronghold'],
            'T': ['Terrace', 'Territory', 'Thicket', 'Timberland', 'Tower', 'Town', 'Trail', 'Trench', 'Tundra', 'Township'],
            'U': ['Undergrowth', 'Underpass', 'Union', 'Uplands', 'Upper', 'Utopia', 'Utterness', 'Umbrage'],
            'V': ['Vale', 'Valley', 'Vault', 'View', 'Villa', 'Village', 'Vineyards', 'Vista', 'Void', 'Vanguard'],
            'W': ['Ward', 'Wasteland', 'Water', 'Way', 'Wetlands', 'Wilds', 'Wood', 'Woods', 'Works', 'Warren'],
            'X': ['Xanadu', 'Xenolith', 'Xerophyte', 'X-Roads', 'Xeric', 'Xyst', 'X-Point', 'X-ing'],
            'Y': ['Yard', 'Yonder', 'Yurt', 'Yards', 'Yielding', 'York', 'Yukon', 'Yews'],
            'Z': ['Zone', 'Zenith', 'Zephyr', 'Zigzag', 'Ziggurat', 'Zion', 'Zodiac', 'Zocalo'],
        }

        # Possessive patterns
        PATTERNS = [
            "{name}'s {place}",      # Karl's Kingdom
            "{place} of {name}",     # Kingdom of Karl
            "{name} {place}",        # Karl Kingdom
        ]

        if not name:
            return ""
        
        first_letter = name[0].upper()
        
        # Get possible descriptors for this letter
        descriptors = PLACE_DESCRIPTORS.get(first_letter, ['Place', 'Point', 'Precinct'])
        
        # Choose descriptor
        if descriptor and descriptor in descriptors:
            place = descriptor
        else:
            place = random.choice(descriptors)

        return place
        
        # Choose pattern
        if pattern is not None and 0 <= pattern < len(PATTERNS):
            template = PATTERNS[pattern]
        else:
            template = random.choice(PATTERNS)
        
        return template.format(name=name, place=place)

    @classmethod
    def cityName(cls, name , pattern=None, descriptor=None, use_suffix=None):
        # Dictionary of place descriptors organized by first letter
        SETTLEMENT_DESCRIPTORS = {
            'A': ['Acres', 'Arbor', 'Ashton', 'Auburn', 'Avon', 'Aldridge', 'Ashford', 'Aston'],
            'B': ['Bay', 'Beach', 'Bridge', 'Brook', 'Burg', 'Borough', 'Bluff', 'Bend'],
            'C': ['City', 'Cove', 'Creek', 'Crest', 'Crossing', 'Center', 'Cape', 'Corners'],
            'D': ['Dale', 'Dell', 'Dunes', 'Down', 'Dock', 'Delta', 'Downs', 'Den'],
            'E': ['End', 'Edge', 'Estates', 'Elms', 'Enclave', 'Evergreen', 'East', 'Elm'],
            'F': ['Falls', 'Field', 'Fields', 'Ford', 'Forest', 'Fort', 'Forks', 'Ferry'],
            'G': ['Glen', 'Glade', 'Green', 'Grove', 'Gate', 'Gardens', 'Groves', 'Gap'],
            'H': ['Harbor', 'Haven', 'Heights', 'Hill', 'Hills', 'Hollow', 'Heath', 'Hurst'],
            'I': ['Isle', 'Island', 'Inlet', 'Inn', 'Ironworks', 'Ivy', 'Isles', 'Inches'],
            'J': ['Junction', 'Jetty', 'Juncture', 'Junction', 'Jamestown', 'Jardin', 'Jct', 'Joya'],
            'K': ['Key', 'Knoll', 'Knolls', 'Keep', 'Keystone', 'Kingswood', 'Kirk', 'Knolle'],
            'L': ['Lake', 'Landing', 'Lawn', 'Ledge', 'Lock', 'Lodge', 'Lagoon', 'Lynn'],
            'M': ['Manor', 'Meadow', 'Meadows', 'Mill', 'Mills', 'Mount', 'Moor', 'Mountain'],
            'N': ['North', 'Nook', 'Narrows', 'Neck', 'Nest', 'Newton', 'New', 'Notch'],
            'O': ['Oaks', 'Orchard', 'Overlook', 'Outpost', 'Outlet', 'Oak', 'Oasis', 'Old'],
            'P': ['Park', 'Pines', 'Plains', 'Point', 'Pond', 'Port', 'Plaza', 'Pass'],
            'Q': ['Quarry', 'Quarter', 'Quay', 'Queen', 'Quarters', 'Quayside', 'Quest', 'Quince'],
            'R': ['Ridge', 'River', 'Rock', 'Run', 'Ranch', 'Rapids', 'Reach', 'Rest'],
            'S': ['Springs', 'Shore', 'Shores', 'South', 'Station', 'Summit', 'Shire', 'Side'],
            'T': ['Town', 'Terrace', 'Trace', 'Trail', 'Township', 'Tower', 'Thicket', 'Timber'],
            'U': ['Union', 'Uplands', 'Upper', 'Underwood', 'Unity', 'University', 'Upton', 'Utopia'],
            'V': ['Vale', 'Valley', 'View', 'Villa', 'Village', 'Vista', 'Ville', 'Vineyards'],
            'W': ['West', 'Water', 'Waters', 'Way', 'Wells', 'Wood', 'Woods', 'Wick'],
            'X': ['Xanadu', 'X-Roads', 'Xing', 'Xavier', 'Xeric', 'Xenia', 'Xenophon', 'Xyst'],
            'Y': ['Yard', 'Yonder', 'York', 'Yards', 'Yew', 'Yews', 'Yale', 'Yarmouth'],
            'Z': ['Zone', 'Zenith', 'Zephyr', 'Zion', 'Zinc', 'Zodiac', 'Zona', 'Zuni'],
        }

        # Naming patterns for settlements
        PATTERNS = [
            "{name}ville",           # Karlville
            "{name}ton",             # Karlton
            "{name}burg",            # Karlburg
            "{name}wood",            # Karlwood
            "{name} {place}",        # Karl Creek
            "{name}'s {place}",      # Karl's Crossing
            "{place} of {name}",     # City of Karl
            "New {name}",            # New Karl
            "Old {name}",            # Old Karl
            "Little {name}",         # Little Karl
            "Upper {name}",          # Upper Karl
            "Lower {name}",          # Lower Karl
            "East {name}",           # East Karl
            "West {name}",           # West Karl
            "North {name}",          # North Karl
            "South {name}",          # South Karl
        ]

        # Shorter patterns for descriptors (avoid double suffixes)
        DESCRIPTOR_PATTERNS = [
            "{name} {place}",        # Karl Creek
            "{name}'s {place}",      # Karl's Crossing
            "{place} of {name}",     # City of Karl
        ]

        if not name:
            return ""
        
        first_letter = name[0].upper()
        
        # Get possible descriptors for this letter
        descriptors = SETTLEMENT_DESCRIPTORS.get(first_letter, ['Place', 'Point', 'Plaza'])
        
        # Decide whether to use suffix or descriptor pattern
        if use_suffix is None:
            use_suffix = random.choice([True, False])
        
        if use_suffix:
            # Use simple suffix patterns (first 7 patterns)
            patterns = PATTERNS[:7]
            if pattern is not None and 0 <= pattern < len(patterns):
                template = patterns[pattern]
            else:
                template = random.choice(patterns)
            
            if "{place}" in template:
                place = descriptor if descriptor and descriptor in descriptors else random.choice(descriptors)
                return template.format(name=name, place=place)
            else:
                return template.format(name=name)
        else:
            # Use directional/size prefix patterns (last 9 patterns)
            patterns = PATTERNS[7:]
            if pattern is not None and 0 <= pattern < len(patterns):
                template = patterns[pattern]
            else:
                template = random.choice(patterns)
            return template.format(name=name)

    @staticmethod
    def decode(s: str) -> 'CountryFlag':
        """Decode CountryFlag from string."""
        parts = s.split('|')
        primary = parts[0]
        name = parts[1]
        year = int(parts[2])
        if len(parts) > 3:
            patternIndex = int(parts[3])
        else:
            patternIndex = 0
        
        # Convert hex back to RGB tuple (0-1 range)
        rgb = plt.matplotlib.colors.to_rgb(primary)
        
        return CountryFlag(rgb, name=name, year=year,patternIndex=patternIndex)

In [ ]:
#| export
@patch
def encode(self: CountryFlag) -> str:
    """Encode CountryFlag to a single line string."""
    # Format: primary|name|year
    return f"{self.primary}|{self.name}|{self.year}|{self.patternIndex}"

In [ ]:
#| export
@patch
def plain(self:CountryFlag,name,width=3):
    return StyleCSS(name,fill=self.primary,stroke=self.comp,stroke_width=width)

@patch
def kingStyle(self:CountryFlag,name,width=2):
    #saturation = 0.7
    style = self.plain(name,width)
    style.name = f"country_{name}"
    #style.properties["fill"] = style.desaturate(saturation).properties["fill"]
    style.properties["opacity"] = 0.7
    return style

@patch
def contrastStyle(self:CountryFlag,name,width=1.5):
    return StyleCSS(
            f"contrast_{name}",
            fill=self.comp,
            stroke="#000",
            stroke_width=width
        )

@patch
def labelStyle(self:CountryFlag,name,width=1):
    return StyleCSS(
            f"contrast_{name}",
            fill=self.tri1,
            stroke="#36454F",
            stroke_width=width
        )

In [ ]:
flags = CountryFlag.seaborn("husl",4)
for flag in flags:
    print(flag.name , flag.capital)


In [ ]:
hexCount = 9
radius = 30
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    style = flat.plain(f"flag_{i}",width=10)
    sampleHex = Hex(radius=30, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
#| export


@patch
def circlePattern(self:CountryFlag, id):
        """Generate a circle pattern definition"""
        content = f'<rect width=120 height= 90 fill="{self.comp}"/><circle cx="{50}" cy="{45}" r="{30}" fill="{self.primary}"/>'
        return SVGDef("pattern", id, content, 
                    width=120, height=90, 
                    patternUnits="userSpaceOnUse")


In [ ]:
#| export
@patch
def triPattern(self:CountryFlag, id):
        """Generate a circle pattern definition"""
        content = f'''<rect width=96 height= 96 fill="{self.primary}"/>
        <circle cx="{48}" cy="{48}" r="{48}" fill="{self.comp}"/>
        <circle cx="{48}" cy="{48}" r="{32}" fill="{self.tri1}"/>
        <circle cx="{48}" cy="{48}" r="{16}" fill="{self.tri2}"/>'''
        
        return SVGDef("pattern", id, content, 
                    width=96, height=96, 
                    patternUnits="userSpaceOnUse")

In [ ]:
#| export
@patch
def swirl(self:CountryFlag,id):

    content = f"""<g  fill='{self.primary}'><rect width=400 height=400 fill='{self.primary}' /></g>
<g  fill='{self.comp}' fill-opacity='1'><path d='M400 58.58c-38.95 0-74.21 15.74-99.79 41.21c-25.61 25.72-61.05 41.64-100.21 41.64s-74.6-15.92-100.21-41.64C74.21 74.32 38.95 58.58 0 58.58c-78.11 0-141.42 63.32-141.42 141.42S-78.11 341.42 0 341.42c38.95 0 74.21-15.74 99.79-41.21c25.61-25.72 61.05-41.64 100.21-41.64s74.6 15.92 100.21 41.64c25.58 25.47 60.84 41.21 99.79 41.21c78.11 0 141.42-63.32 141.42-141.42S478.11 58.58 400 58.58z'/><circle  cx='200' cy='1' r='60'/><circle  cx='200' cy='400' r='60'/></g><g  fill='{self.primary}'><circle  cx='0' cy='200' r='60'/><circle  cx='400' cy='200' r='60'/></g>
"""
    pat = SVGDef("pattern", id, content,
                  width=400, height=400,
                  patternUnits="userSpaceOnUse")
    scale = 0.2
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.swirl(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
#| export
@patch
def yin(self:CountryFlag,id):
    content = f"""<rect fill='{self.primary}' width='800' height='800'/>
     	<g id="YinYang" fill="{self.comp}" stroke="none" stroke-width="0" fill-rule="evenodd">
		<title>Yin-Yang, by Adam Stanislav</title>
		<desc>The entire graphic is drawn as a single path filled with black (or any other color you change the value of “fill” in line 4). The other half, usually shown in white is created here as a hole in the path. That means it is completely transparent, and has whatever color its background has. To achieve this, not just with SVG but with other vector formats, any black portion of the path is drawn counterclockwise, any “white” portion clockwise. Also, this graphic is taking advantage of the kappa constant described in my e-book Bézier Circles and other shapes, freely downloadable from https://www.smashwords.com/books/view/483578 .</desc>

		<!-- Note to self: Relative Bézier differences (“c”) are differences of a point from the STARTING point of the curve segment, not from the most recent point. -->
		<path d="M400 0C179.086 0 0 179.086 0 400 0 620.914 179.086 800 400 800 620.914 800 800 620.914 800 400 800 179.086 620.914 0 400 0zM400 10C184.609 10 10 184.609 10 400 10 615.391 184.609 790 400 790 292.304 790 205 682.304 205 600 205 492.3 292.304 400 400 400 507.7 400 600 292.304 600 200 600 92.304 507.7 10 400 10zM400 665c35.895 0 65-29.105 65-65 0-35.895-29.105-65-65-65-35.895 0-65 29.105-65 65 0 35.895 29.105 65 65 65zM400 132c-37.555 0-68 30.445-68 68 0 37.555 30.445 68 68 68 37.555 0 68-30.445 68-68 0-37.555-30.445-68-68-68z" stroke='black' stroke-width='25' stroke-opacity='1' stroke-linecap='square' />
	</g>
  """


    pat = SVGDef("pattern", id, content,
                  width=800, height=800,
                  patternUnits="userSpaceOnUse")
    scale = 0.1 
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.yin(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

How do I fill black in yin

In [ ]:
#| export
@patch
def weave(self:CountryFlag, id):
    
    content = f"""<rect fill='{self.primary}' width='600' height='600'/><path  fill='none' stroke='{self.comp}' stroke-width='55' stroke-opacity='1' stroke-linecap='square' d='M250.5 50.5h-200M250.5 150.5h-200M250.5 250.5h-200M550.5 350.5h-200M550.5 450.5h-200M550.5 550.5h-200M250.5 550.5v-200M150.5 550.5v-200M50.5 550.5v-200M550.5 250.5v-200M450.5 250.5v-200M350.5 250.5v-200'/>
    """
    pat = SVGDef("pattern", id, content,
                  width=600, height=600,
                  patternUnits="userSpaceOnUse")
    scale = 0.25
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
hexCount = 19
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("magma", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.weave(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.yin(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.triPattern(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
for pat in canvas.definitions:
    print(pat.attributes["id"])

In [ ]:


hexStyles = CountryFlag.seaborn("Accent", hexCount)
hexIndex = 3
name = f"CountrySwirl_{hexIndex}"
patternName = f"{name}_pat"
hexPat = hexStyles[hexIndex].swirl(patternName)

#some drawing setup 
hexStyle = StyleCSS(name, fill=f"url(#{patternName})")
canvas = SVGBuilder()
canvas.width=200 ;canvas.height=200
canvas.add_style(hexStyle)
canvas.add_definition(hexPat)

# add our hex to the canvas
sampleHex = Hex(radius=50,center=MapCord(100,100),style=hexStyle)
canvas.adjust("main",sampleHex.svg())

#show our work
canvas.show()
canvas.adjust("main",sampleHex.svg())

#show our work
canvas.show()



In [ ]:
#| export
@patch
def sheridan(self:CountryFlag,id):
    content = f"""<rect fill='{self.primary} width='1600' height='900'/><path  fill='{self.tri1}' d='M799 0v0.5c0 0-49.4 40.5-49.4 90s99.8 130.3 99.8 180c0 49.7-99.8 130.5 -99.8 180s99.8 130.3 99.8 180s-99.8 130.5-99.8 180c0 46.5 40.4 89.5 40.4 89.5h9h10h792v-900h-802z'/><path  fill='{self.tri2}' d='M751.6 450.5c0-49.5 99.8-130.3 99.8-180c0-49.7-99.8-130.5-99.8-180s49.4-90 49.4-90v-0.5h-802v900h793c0 0-40.4-43-40.4-89.5c0-49.5 99.8-130.3 99.8-180s-99.8-130.5-99.8-180z'/>"""
    pat = SVGDef("pattern", id, content,
                  width=1600, height=900,
                  patternUnits="userSpaceOnUse")
    scale = 0.1
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat

Lets redo sheridan so it is vertical waves of comp and primary

It does look good as a repeated pattern. Can you build a better version?

can use angled''' <svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 2000 1500'><rect fill='#ffffff' width='2000' height='1500'/><pattern id='p' width='2000' height='1000' patternUnits='userSpaceOnUse' patternTransform=''><path  fill='#484B57' d='M1500 100l-100 100v200l200 200V200zm500 100V0l-200 200v200zM2000 1000l-144-144 144-56zm-200-600l-400 400 200 200V800l400-400zM1200 800v200h-200zm100-300l-300 300h200l200-200zm-200-400L1200 0h-200v200l200 200h200z'/><path  fill='#D3E0D9' d='M1600 0h-200l200 200v400l200-200V200h-200L1800 0zM2000 600V400l-200 200v200zM800 0H600l200 200 400 400 100-100zm400 0l-100 100 300 300V200zm200 800h-200l200 200h200zM1800 800l-200 200h200l200-200zm-800 0L600 400H400l400 400-200 200h200l200-200zM600 800L400 600V400L200 600 0 800v200h400l100-100zm0-800H200l200 200z'/><path  fill='#484B57' d='M800 200V0L400 400h200l100-100 300 300h200zM0 200V0h200zm600 400L400 400v200l200 200h200zM400 200L300 100 0 400l200 200V400zm100 700L200 600 100 700l300 300zm-300 100H0V800z'/></pattern><rect fill='url(#p)' width='100%' height='100%'/></svg> ``` as th basis of a pattern this one has 3 colors

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.sheridan(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

So we have a bunch of flag patterns (swirl, yin, sheridan), but ultimately we need a dispatch function using the patternIndex for a flag. Can you build? any other interesing kind patterns to use

In [ ]:
#| export
@patch
def chevron(self:CountryFlag, id):
    """Zigzag chevron pattern"""
    content = f"""<rect fill='{self.primary}' width='200' height='200'/>
    <path fill='{self.comp}' d='M0 0 L100 50 L200 0 L200 50 L100 100 L0 50Z'/>
    <path fill='{self.comp}' d='M0 100 L100 150 L200 100 L200 150 L100 200 L0 150Z'/>"""
    pat = SVGDef("pattern", id, content, width=200, height=200, patternUnits="userSpaceOnUse")
    pat.attributes['patternTransform'] = 'scale(0.25)'
    return pat

@patch
def scales(self:CountryFlag, id):
    """Fish-scale / overlapping semicircle pattern"""
    content = f"""<rect fill='{self.primary}' width='100' height='100'/>
    <circle cx='50' cy='100' r='50' fill='none' stroke='{self.comp}' stroke-width='12'/>
    <circle cx='0' cy='50' r='50' fill='none' stroke='{self.comp}' stroke-width='12'/>
    <circle cx='100' cy='50' r='50' fill='none' stroke='{self.comp}' stroke-width='12'/>"""
    pat = SVGDef("pattern", id, content, width=100, height=100, patternUnits="userSpaceOnUse")
    pat.attributes['patternTransform'] = 'scale(0.35)'
    return pat

@patch
def diamonds(self:CountryFlag, id):
    """Argyle diamond grid"""
    content = f"""<rect fill='{self.primary}' width='200' height='200'/>
    <polygon points='100,0 200,100 100,200 0,100' fill='{self.comp}'/>
    <line x1='0' y1='0' x2='200' y2='200' stroke='{self.tri1}' stroke-width='4'/>
    <line x1='200' y1='0' x2='0' y2='200' stroke='{self.tri1}' stroke-width='4'/>"""
    pat = SVGDef("pattern", id, content, width=200, height=200, patternUnits="userSpaceOnUse")
    pat.attributes['patternTransform'] = 'scale(0.2)'
    return pat


In [ ]:
#| export
@patch
def fanBlade(self:CountryFlag, id):
    blade_id = f"{id}_b"
    content = f"""<rect width='100' height='100' fill='{self.primary}'/>
    <g stroke='none'>
        <path id='{blade_id}' fill='{self.comp}' d='M75 50c-6.9 0-12.5-5.6-12.5-12.5S68.1 25 75 25c0-6.9-5.6-12.5-12.5-12.5S50 18.1 50 25s-5.6 12.5-12.5 12.5S25 31.9 25 25c-6.9 0-12.5 5.6-12.5 12.5S18.1 50 25 50s12.5 5.6 12.5 12.5S31.9 75 25 75c0 6.9 5.6 12.5 12.5 12.5S50 81.9 50 75s5.6-12.5 12.5-12.5S75 68.1 75 75c6.9 0 12.5-5.6 12.5-12.5S81.9 50 75 50z'/>
        <use href='#{blade_id}' x='-50' y='50'/>
        <use href='#{blade_id}' x='-50' y='-50'/>
        <use href='#{blade_id}' x='50' y='-50'/>
        <use href='#{blade_id}' x='50' y='50'/>
    </g>"""

    pat = SVGDef("pattern", id, content,
                  width=100, height=100,
                  patternUnits="userSpaceOnUse")
    scale = 0.4
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
#| export
@patch
def sheridan(self:CountryFlag, id):
    W, H = 200, 200
    A = 30  # wave amplitude
    cx1, cx2 = 50, 150  # wave boundary centers

    content = f"""<rect width='{W}' height='{H}' fill='{self.primary}'/>
    <path fill='{self.comp}' d='
        M {cx1},0
        Q {cx1+A},{H//4} {cx1},{H//2}
        Q {cx1-A},{3*H//4} {cx1},{H}
        L {cx2},{H}
        Q {cx2-A},{3*H//4} {cx2},{H//2}
        Q {cx2+A},{H//4} {cx2},0
        Z'/>"""

    pat = SVGDef("pattern", id, content,
                  width=W, height=H,
                  patternUnits="userSpaceOnUse")
    scale = 0.5
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
#| export
@patch
def leafy(self:CountryFlag, id):
    content = f"""<rect fill='{self.primary}' width='250' height='200'/>
    <g fill='{self.comp}'>
        <path d='M161 100c0 33.22-36 34.14-36 60c0-25.86-36-26.78-36-60s36-34.14 36-60C125 65.86 161 66.78 161 100z'/>
        <path d='M35 0C35 33.22 0 34.14 0 60c0-25.86-65-90-65-90S35-33.22 35 0z'/>
        <path d='M35 200c0 33.22-100 30-100 30s65-64.14 65-90C0 165.86 35 166.78 35 200z'/>
        <path d='M315-30c0 0-65 64.14-65 90c0-25.86-35-26.78-35-60S315-30 315-30z'/>
        <path d='M315 230c0 0-100 3.22-100-30s35-34.14 35-60C250 165.86 315 230 315 230z'/>
    </g>
    <g fill='none' stroke='{self.tri1}' stroke-width='40'>
        <path d='M39.53-98.77c0 40.43 46.61 56.95 46.61 98.85s-47.19 58.59-47.19 99.3s47.19 57.24 47.19 99.3s-46.61 58.41-46.61 98.85'/>
        <path d='M211.72 297.53c0-40.71-47.19-57.24-47.19-99.3s46.61-58.41 46.61-98.85c0-40.43-46.61-56.95-46.61-98.85s47.19-58.59 47.19-99.3'/>
    </g>"""

    pat = SVGDef("pattern", id, content,
                  width=250, height=200,
                  patternUnits="userSpaceOnUse")
    scale = 0.35
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
#| export
@patch
def angled(self:CountryFlag, id):
    content = f"""<rect fill='{self.primary}' width='2000' height='1000'/>
<path fill='{self.comp}' d='M1500 100l-100 100v200l200 200V200zm500 100V0l-200 200v200zM2000 1000l-144-144 144-56zm-200-600l-400 400 200 200V800l400-400zM1200 800v200h-200zm100-300l-300 300h200l200-200zm-200-400L1200 0h-200v200l200 200h200z'/>
<path fill='{self.tri1}' d='M1600 0h-200l200 200v400l200-200V200h-200L1800 0zM2000 600V400l-200 200v200zM800 0H600l200 200 400 400 100-100zm400 0l-100 100 300 300V200zm200 800h-200l200 200h200zM1800 800l-200 200h200l200-200zm-800 0L600 400H400l400 400-200 200h200l200-200zM600 800L400 600V400L200 600 0 800v200h400l100-100zm0-800H200l200 200z'/>
<path fill='{self.comp}' d='M800 200V0L400 400h200l100-100 300 300h200zM0 200V0h200zm600 400L400 400v200l200 200h200zM400 200L300 100 0 400l200 200V400zm100 700L200 600 100 700l300 300zm-300 100H0V800z'/>"""

    pat = SVGDef("pattern", id, content,
                  width=2000, height=1000,
                  patternUnits="userSpaceOnUse")
    scale = 0.08
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
def _settlement_pattern(pat_id: str, flag: 'CountryFlag', size: int = 40) -> str:
    """Generate an SVG <pattern> element from a CountryFlag's colors + patternIndex."""
    s = size
    h = s // 2
    p, c, t1, t2 = flag.primary, flag.comp, flag.tri1, flag.tri2
    idx = flag.patternIndex % 6

    if idx == 0:    # horizontal bicolor
        inner = (f'<rect width="{s}" height="{h}" fill="{p}"/>'
                 f'<rect y="{h}" width="{s}" height="{h}" fill="{c}"/>')
    elif idx == 1:  # vertical bicolor
        inner = (f'<rect width="{h}" height="{s}" fill="{p}"/>'
                 f'<rect x="{h}" width="{h}" height="{s}" fill="{c}"/>')
    elif idx == 2:  # tricolor horizontal
        t = s // 3
        inner = (f'<rect width="{s}" height="{t}" fill="{p}"/>'
                 f'<rect y="{t}" width="{s}" height="{t}" fill="{t1}"/>'
                 f'<rect y="{2*t}" width="{s}" height="{t}" fill="{c}"/>')
    elif idx == 3:  # quarters
        inner = (f'<rect width="{h}" height="{h}" fill="{p}"/>'
                 f'<rect x="{h}" width="{h}" height="{h}" fill="{c}"/>'
                 f'<rect y="{h}" width="{h}" height="{h}" fill="{t1}"/>'
                 f'<rect x="{h}" y="{h}" width="{h}" height="{h}" fill="{t2}"/>')
    elif idx == 4:  # cross on solid
        w = s // 5
        inner = (f'<rect width="{s}" height="{s}" fill="{p}"/>'
                 f'<rect x="{h - w//2}" width="{w}" height="{s}" fill="{c}"/>'
                 f'<rect y="{h - w//2}" width="{s}" height="{w}" fill="{c}"/>')
    else:           # diagonal split
        inner = (f'<rect width="{s}" height="{s}" fill="{p}"/>'
                 f'<polygon points="0,0 {s},0 0,{s}" fill="{c}"/>')

    return (f'<pattern id="{pat_id}" width="{s}" height="{s}" '
            f'patternUnits="userSpaceOnUse">{inner}</pattern>')


In [ ]:
#| export
@patch
def bicolorH(self:CountryFlag, id):
    """Horizontal bicolor flag"""
    s = 40; h = s // 2
    content = f'<rect width="{s}" height="{h}" fill="{self.primary}"/><rect y="{h}" width="{s}" height="{h}" fill="{self.comp}"/>'
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")

@patch
def bicolorV(self:CountryFlag, id):
    """Vertical bicolor flag"""
    s = 40; h = s // 2
    content = f'<rect width="{h}" height="{s}" fill="{self.primary}"/><rect x="{h}" width="{h}" height="{s}" fill="{self.comp}"/>'
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")

@patch
def tricolorH(self:CountryFlag, id):
    """Horizontal tricolor flag"""
    s = 42; t = s // 3
    content = (f'<rect width="{s}" height="{t}" fill="{self.primary}"/>'
               f'<rect y="{t}" width="{s}" height="{t}" fill="{self.tri1}"/>'
               f'<rect y="{2*t}" width="{s}" height="{t}" fill="{self.comp}"/>')
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")

@patch
def quarters(self:CountryFlag, id):
    """Quartered flag"""
    s = 40; h = s // 2
    content = (f'<rect width="{h}" height="{h}" fill="{self.primary}"/>'
               f'<rect x="{h}" width="{h}" height="{h}" fill="{self.comp}"/>'
               f'<rect y="{h}" width="{h}" height="{h}" fill="{self.tri1}"/>'
               f'<rect x="{h}" y="{h}" width="{h}" height="{h}" fill="{self.tri2}"/>')
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")


In [ ]:
#| export

@patch
def crossFlag(self:CountryFlag, id):
    """Cross on solid background with center circle in tri2"""
    s = 40; h = s // 2; w = s // 5; r = w 
    content = (f'<rect width="{s}" height="{s}" fill="{self.primary}"/>'
               f'<rect x="{h - w//2}" width="{w}" height="{s}" fill="{self.comp}"/>'
               f'<rect y="{h - w//2}" width="{s}" height="{w}" fill="{self.comp}"/>'
               f'<circle cx="{h}" cy="{h}" r="{r}" fill="{self.tri2}"/>')
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")


In [ ]:
!cat ../../HexMagic/styles.py

In [ ]:
#| export
@patch
def flagPattern(self:CountryFlag, id, scale=None):
    """Dispatch to pattern based on patternIndex, with optional scale override"""
    patterns = [
        self.circlePattern,  # 0
        self.triPattern,     # 1
        self.swirl,          # 2
        self.yin,            # 3
        self.weave,          # 4
        self.sheridan,       # 5
        self.chevron,        # 6
        self.scales,         # 7
        self.diamonds,       # 8
        self.fanBlade,       # 9
        self.leafy,          # 10
        self.bicolorH,       # 11
        self.bicolorV,       # 12
        self.tricolorH,      # 13
        self.quarters,       # 14
        self.crossFlag,      # 15
    ]
    idx = self.patternIndex % len(patterns)
    pat = patterns[idx](id)
    
    if scale is not None:
        # Parse existing scale from patternTransform (default 1.0 if not set)
        existing_transform = pat.attributes.get('patternTransform', 'scale(1)')
        m = re.search(r'scale\(([\d.]+)\)', existing_transform)
        base_scale = float(m.group(1)) if m else 1.0
        pat.attributes['patternTransform'] = f'scale({base_scale * scale})'
    
    return pat


Can we update flagPattern to take a scale parameters. sometimes we are going to need to shrink the flag

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! The simplest approach is to have `flagPattern` accept a `scale` parameter and apply it to the returned pattern's `patternTransform` attribute after the individual pattern method creates it. That way you don't need to modify each pattern method individually.

Something like:

```python
@patch
def flagPattern(self:CountryFlag, id, scale=None):
```

Then after getting the pattern from the dispatch, if `scale` is not None, set `pat.attributes['patternTransform'] = f'scale({scale})'` on the result before returning it.

This will override whatever default scale the individual pattern sets. Want me to write the full updated method, or do you want to give it a go?

So the tricky part is that each of the underlying flags have their own scale. it needs to read the existing scale and then modify it

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good point! So you'll need to:

1. Get the pattern from the dispatch as before
2. Parse the existing `patternTransform` from `pat.attributes` (if it exists) to extract the current scale value
3. Multiply it by the incoming `scale` factor
4. Write back the updated `patternTransform`

Note that some patterns (like `circlePattern`, `bicolorH`, etc.) don't set a `patternTransform` at all — so you'd treat those as having an implicit scale of `1.0`.

You can pull the existing scale out with something like:

```python
existing = pat.attributes.get('patternTransform', 'scale(1)')
```

Then parse the number out of that string, multiply, and set it back. Want to give it a try?

## Packets

Can you make the gallery have 3 rows

In [ ]:
hexCount = 16
radius = 80
padding = 20
itemWidth = (radius * 2 + padding)
cols = 6
rowHeight = radius * 2 + 60  # hex height + label space

canvas = SVGBuilder()
canvas.width = cols * itemWidth
canvas.height = 3 * rowHeight + padding

pattern_names = [
    "circle", "tri", "swirl", "yin", "weave",
    "sheridan", "chevron", "scales", "diamonds", "fanBlade", "leafy",
    "bicolorH", "bicolorV", "tricolorH", "quarters", "crossFlag"
]

flag = CountryFlag.seaborn("husl", 1)[0]

for i in range(hexCount):
    flag.patternIndex = i
    name = f"gallery_{i}"
    patternName = f"{name}_pat"
    
    row = i // cols
    col = i % cols
    
    pat = flag.flagPattern(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    cx = (padding + radius) + col * itemWidth
    cy = padding + radius + row * rowHeight
    
    sampleHex = Hex(radius=radius, center=MapCord(cx, cy), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pat)
    
    label = f'<text x="{cx}" y="{cy + radius + 20}" text-anchor="middle" font-size="14" font-family="sans-serif" fill="#333">{pattern_names[i]}</text>'
    canvas.adjust(f"label-{i}", label)

canvas.show()


In [ ]:
class Resources(Enum):
    """Available Foods for pieces"""
    GRAIN = "GRAIN"
    CHICKEN = "CHICKEN"
    VEGATABLES = "VEGATABLES"
    FRUIT = "FRUIT"
    FISH = "FISH"
    HERD = "HERD"
    

## Piece of mind

In [ ]:
#!cat ../../docs/*.md

from dataclasses import dataclass, field
from typing import Optional, List, Set
from enum import Enum
import uuid

class PieceGoal(Enum):
    """Available goals for pieces"""
    EXPLORE = "explore"
    HARVEST = "harvest"
    ATTACK = "attack"
    MOVE = "move"
    SETTLE = "settle"
    DEFEND = "defend"
    PATROL = "patrol"

class PersonalityTrait(Enum):
    """Personality tendencies when acting autonomously"""
    AGGRESSIVE = "aggressive"      # Prefers attack/explore
    DEFENSIVE = "defensive"        # Prefers defend/settle
    ECONOMIC = "economic"          # Prefers harvest/settle
    EXPLORER = "explorer"          # Prefers explore/move
    BALANCED = "balanced"          # No strong preference




@dataclass
class Piece:
    """A game piece representing a group of units."""
    
    # Identity
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    owner_id: int = 0  # Kingdom/country ID
    parent_id: Optional[str] = None  # ID of piece that spawned this
    
    # Core attributes
    size: int = 100  # Number of units in this piece
    health: int = 100  # Hit points (0-100 scale)
    max_health: int = 100
    
    # Vision and intelligence
    sight: int = 3  # How many hex rings they can see
    memory: float = 0.8  # Retention rate (0-1, higher = better memory)
    
    # Movement
    movement_range: int = 4  # Max weighted hexes per turn
    current_position: int = -1  # Current hex index
    
    # Goals and targeting
    goal: PieceGoal = PieceGoal.EXPLORE
    target_position: Optional[int] = None  # Target hex index
    target_piece: Optional[str] = None  # Target piece UUID
    
    # Behavior
    personality: PersonalityTrait = PersonalityTrait.BALANCED
    
    # Relationships
    spawned_pieces: List[str] = field(default_factory=list)  # UUIDs of children
    
    # Knowledge (what hexes this piece knows about)
    known_hexes: Set[int] = field(default_factory=set)
    knowledge_freshness: dict = field(default_factory=dict)  # hex_idx -> turn_last_seen
    
    # Settlement state
    is_settled: bool = False
    settle_progress: int = 0  # Turns spent settling (settlement complete at threshold)
    settle_threshold: int = 3


In [ ]:
import os
import glob

def concatenate_text_files(directory_path):
    # Change the current working directory to the specified path
    os.chdir(directory_path)
    
    # Use glob to find all files matching a pattern (e.g., all .txt files)
    # Adjust '*.txt' to match your specific file extension if needed
    file_list = glob.glob('*.py') 
    
    # Open the output file in write mode ('w')
    ret = ""
    for fname in file_list:
        print(fname)
        with open(fname, 'r') as infile:
            # Read the content and write it to the output file
            ret += f"{fname}\n\n " + infile.read() 
            # Optional: Add a newline character as a separator between files
            #outfile.write('\n')
    return ret

In [ ]:
primitives = concatenate_text_files("../../HexMagic/plot")

does $`primitives` have information about HexRegions

In [ ]:
rootIdeas  = concatenate_text_files("../../HexMagic")

in $`rootIdeas` do you see how terrain has encode and decode. can we build something similar for piece

In [ ]:
from HexMagic.core import Terrain

In [ ]:
??Terrain.encode

Does the terrain style encoding make sense to work for piece

So we have a bunch of patterns for the flag ( sheridan, weave, yin, swirl, triPattern ) but ultimately the country should just have one pattern. can we build a function that will assign one pattern. This will need to be consistent (ie encode and decode) so maybe it is an int index into a function 

Can you write it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def flagPattern(self:CountryFlag, id, scale=None):
    """Dispatch to pattern based on patternIndex, with optional scale override"""
    patterns = [
        self.circlePattern,  # 0
        self.triPattern,     # 1
        self.swirl,          # 2
        self.yin,            # 3
        self.weave,          # 4
        self.sheridan,       # 5
        self.chevron,        # 6
        self.scales,         # 7
        self.diamonds,       # 8
        self.fanBlade,       # 9
        self.leafy,          # 10
        self.bicolorH,       # 11
        self.bicolorV,       # 12
        self.tricolorH,      # 13
        self.quarters,       # 14
        self.crossFlag,      # 15
    ]
    idx = self.patternIndex % len(patterns)
    pat = patterns[idx](id)
    
    if scale is not None:
        # Parse existing scale from patternTransform (default 1.0 if not set)
        existing_transform = pat.attributes.get('patternTransform', 'scale(1)')
        import re
        m = re.search(r'scale\(([\d.]+)\)', existing_transform)
        base_scale = float(m.group(1)) if m else 1.0
        pat.attributes['patternTransform'] = f'scale({base_scale * scale})'
    
    return pat
```

This grabs whatever `patternTransform` the individual pattern method already set (e.g. `scale(0.2)` for swirl), parses the number out, multiplies by your `scale` factor, and writes it back. Patterns that don't set a transform at all get treated as `scale(1)`.